# 3. Data Pre-processing

---
🧠 *This is a learning version with code cells emptied. Read the instructions, write the code yourself. Peek at the original if stuck.*

Data pre-processing techniques generally refer to the addition, deletion, or transformation of training set data. Different models have different sensitivities to the type of predictors in the model; *how* the predictors enter the model is also important.

The need for data pre-processing is determined by the type of model being used. Some procedures, such as tree-based models, are notably insensitive to the characteristics of the predictor data. Others, like linear regression, are not. In this chapter, a wide array of possible methodologies are discussed.

How the predictors are encoded, called *feature engineering*, can have a significant impact on model performance. Often the most effective encoding of the data is informed by the modeler's understanding of the problem and thus is not derived from any mathematical techniques.

🧠 **Chapter 3 is all about: preparing your data BEFORE modeling.** Garbage in = garbage out. The transformations you apply to predictors can make or break your model.

- **Section 3.2**: Transform individual predictors (centering, scaling, fixing skewness with Box-Cox)
- **Section 3.3**: Transform groups of predictors (spatial sign for outliers, PCA for dimensionality reduction)
- **Section 3.4**: Handle missing values (KNN imputation, linear model imputation)
- **Section 3.5**: Remove problematic predictors (near-zero variance, highly correlated predictors)

## 3.1 Case Study: Cell Segmentation in High-Content Screening

Check if data exists.

In [ ]:
# 🧠 TIP: Same pattern as Chapter 2 — shell command to list files.
# TODO: list files in ../datasets/segmentationOriginal/

!ls -l ../datasets/segmentationOriginal/

This dataset is from Hill et al. (2007) that consists of 2019 cells. Of these cells, 1300 were judged to be poorly segmented (PS) and 719 were well segmented (WS); 1009 cells were reserved for the training set.

🧠 **Binary classification problem:** Predict PS vs WS (poorly vs well segmented cells). This is a supervised learning task where the target is categorical (two classes).

In [ ]:
# 🧠 TIP: pandas + numpy are the standard data science libraries.
#   pd.read_csv() loads the CSV into a DataFrame.
#   The dataset has 2019 rows and 120 columns — that's a lot of features!
#   This is where pre-processing becomes critical.
#
# TODO: import numpy as np and pandas as pd
# TODO: read '../datasets/segmentationOriginal/segmentationOriginal.csv'
#       into a variable called 'cell_segmentation'




In [ ]:
# 🧠 TIP: .shape gives (rows, columns).
#   (2019, 120) means 2019 cells and 120 columns.
#   The 'Class' column is our target (PS or WS).
#   The rest are features describing cell morphology.
#
# TODO: print the shape of cell_segmentation



A first look at the dataset.

In [ ]:
# 🧠 TIP: .head() shows the first 5 rows.
#   Notice the columns: 'Cell' (ID), 'Case' (Train/Test),
#   'Class' (PS/WS — our target!), and 117 predictor columns.
#   Predictor names follow patterns like:
#     - AngleCh1, AreaCh1, AvgIntenCh1, AvgIntenCh2, etc.
#     - The 'Ch' refers to different imaging channels.
#     - 'Status' columns seem to be categorical flags.
#
# TODO: display the first 5 rows of cell_segmentation



This chapter will use the training set samples to demonstrate data pre-processing techniques.

In [ ]:
# 🧠 TIP: The 'Case' column tells us which rows are Train vs Test.
#   We split the data using this column.
#   
#   NOTE: The original code uses .ix[] which is DEPRECATED in newer pandas.
#   Use .loc[] instead for label-based indexing.
#
#   We'll also use .groupby('Case').count() to verify the split.
#
# TODO: group cell_segmentation by 'Case' and call .count()
#       to see how many Train vs Test rows there are



In [ ]:
# 🧠 TIP: Now filter the data to get only training rows.
#   Use .loc[] with a boolean condition:
#     cell_segmentation['Case'] == 'Train'
#
#   Then do the same for Test rows.
#   Display the first 5 rows of the training set to verify.
#
# TODO: filter cell_segmentation where Case == 'Train' into cell_train
# TODO: filter where Case == 'Test' into cell_test
# TODO: display cell_train.head(5)




## 3.2 Data Transformation for Individual Predictors

Transformations of predictor variables may be needed for several reasons. Some modeling techniques may have strict requirements, such as the predictors having a common scale. In other cases, creating a good model may be difficult due to specific characteristics of the data (e.g., outliers).

🧠 **Why transform individual predictors?**
- Some models (like linear regression, SVM, PCA) assume predictors are on similar scales.
- A predictor with values in the thousands will dominate one with values in 0-1.
- Skewed distributions can make models behave poorly.
- Outliers can pull regression lines away from the true pattern.

### Centering and Scaling

To center a predictor variable, the average predictor value is subtracted from all the values. As a result of centering, the predictor has a zero mean. Similarly, to scale the data, each value of the predictor variable is divided by its standard deviation. Scaling the data coerce the values to have a common standard deviation of one. These manipulations are generally used to improve the numerical stability of some calculations, such as PLS. The only real downside to these transformations is a loss of interpretability of the individual values.

🧠 **Standardization formula:**
  $$z = \frac{x - \mu}{\sigma}$$
  - Subtract mean ($\mu$) → centered at 0
  - Divide by standard deviation ($\sigma$) → unit variance
  
  Use `sklearn.preprocessing.scale()` or `StandardScaler`.
  
  **Key rule: Fit on TRAINING data, transform BOTH training and test data.**
  Never fit on the test set — that would be data leakage!

### Transformations to Resolve Skewness

An un-skewed distribution is one that is roughly symmetric. A rule of thumb to consider is that skewed data whose ratio of the highest value to the lowest value is greater than 20 have significant skewness. The sample skewness statistic is defined $$\text{skewness} = {\sum (x_i - \bar{x})^3 \over (n - 1) v^{3/2}},$$ where $$v = {\sum (x_i - \bar{x})^2 \over (n - 1)}.$$ Note that the skewness for a normal distribution is zero.

🧠 **Skewness intuition:**
- Right-skewed (positive): long tail on the right. Many small values, few large ones.
- Left-skewed (negative): long tail on the left. Many large values, few small ones.
- Log transformation ($log(x)$) is great for right-skewed positive data.
- Square root ($\sqrt{x}$) is milder than log.
- Box-Cox generalizes both and finds the best transformation automatically!

The cell segmentation data contain a predictor that measures the standard deviation of the intensity of the pixels in the actin filaments.

In [ ]:
# 🧠 TIP: matplotlib setup — same as Chapter 2.
#
# TODO: set up matplotlib inline, import pyplot as plt
# TODO: set figure size to (10, 7.5) and enable grid




In [ ]:
# 🧠 TIP: We'll look at the distribution of 'VarIntenCh3'.
#   This measures variance of pixel intensity in channel 3.
#   We'll plot histograms in natural units, log units, and sqrt units.
#
#   plt.subplots(1, 3) creates 1 row, 3 columns of plots side by side.
#
#   np.log() applies natural log to every value.
#   np.sqrt() applies square root.
#
#   bins=20 means 20 bars in the histogram.
#
# TODO: create 3 histograms of VarIntenCh3: natural, log, sqrt
# TODO: label the x-axis and y-axis appropriately






The histogram shows a strong right skewness. The log transformation seems to work well for this dataset. The ratio of the smallest to largest value and the sample skewness statistic all agree with the histogram under natural units.

🧠 **What you should see:** Natural units is heavily right-skewed (bunched up on the left). Log makes it look almost symmetric (bell-shaped). Square root helps but not as much as log.

In [ ]:
# 🧠 TIP: scipy.stats.skew() computes the sample skewness.
#   - Skewness ≈ 0 → symmetric
#   - Skewness > 0 → right-skewed
#   - Skewness < 0 → left-skewed
#
#   np.max() / np.min() gives the ratio rule-of-thumb.
#   Ratio > 20 suggests significant skewness.
#
#   NOTE: This notebook uses Python 2 syntax (print without parentheses).
#   In Python 3, use print() with parentheses.
#
# TODO: import skew from scipy.stats
# TODO: compute max/min ratio of VarIntenCh3
# TODO: compute sample skewness of VarIntenCh3
# TODO: print both values





Alternatively, statistical models can be used to empirically identify an appropriate transformation. One of the most famous transformations is the Box-Cox family, i.e.
\begin{equation}
x^* = \begin{cases} {x^{\lambda}-1 \over \lambda} & \text{if} \ \lambda \neq 0 \\ log(x) & \text{if} \ \lambda = 0 \end{cases}
\end{equation}
This family covers the log ($\lambda = 0$), square ($\lambda = 2$), square root ($\lambda = 0.5$), inverse ($\lambda = -1$), and others in-between. Using the training data, $\lambda$ can be estimated using maximum likelihood estimation (MLE). This procedure would be applied independently to each predictor data that contain values **greater than 0**.

🧠 **Box-Cox summary:**
- Finds the optimal $\lambda$ automatically.
- Log is just a special case ($\lambda = 0$).
- Only works for **positive** data.
- Returns both the transformed data and the estimated $\lambda$.

The boxcox() in *scipy.stats* finds the estimated lambda and performs the transformation at the same time.

In [ ]:
# 🧠 TIP: scipy.stats.boxcox() does two things at once:
#   1. Estimates the optimal lambda (using MLE)
#   2. Transforms the data using that lambda
#
#   It returns a tuple: (transformed_data, estimated_lambda)
#
#   An estimated lambda of ~0.12 is close to 0 (log transformation).
#   This confirms that log was a good choice!
#
# TODO: import boxcox from scipy.stats
# TODO: apply boxcox to VarIntenCh3 and print the estimated lambda




Take another predictor for example.

In [ ]:
# 🧠 TIP: Same pattern — compare original vs Box-Cox transformed.
#   'PerimCh1' is a different predictor (perimeter of cell in channel 1).
#   boxcox()[0] gives the transformed data.
#   boxcox()[1] gives the estimated lambda.
#
# TODO: create 2 histograms side by side: original and Box-Cox transformed
#       of cell_train['PerimCh1']





## 3.3 Data Transformations for Multiple Predictors

These transformations act on groups of predictors, typically the entire set under consideration. Of primary importance are methods to resolve outliers and reduce the dimension of the data.

🧠 **Now we move from individual transformations to group transformations.** These techniques consider ALL predictors at once.

### Transformations to Resolve Outliers

We generally define outliers as samples that are exceptionally far from the mainstream of the data. Even with a thorough understanding of the data, outliers can be hard to define. However, we can often identify an unusual value by looking at a figure. When one or more samples are suspected to be outliers, the first step is to make sure that the values are scientifically valid and that no data recording errors have occurred. Great care should be taken not to hastily remove or change values, especially if the sample size is small. With small sample sizes, apparent outliers might be a result of a skewed distribution where there are not yet enough data to see the skewness. Also, the outlying data may be an indication of a special part of the population under study that is just starting to be sampled. Depending on how the data were collected, a "cluster" of valid points that reside outside the mainstream of the data might belong to a different population than the other samples, e.g. *extrapolation* and *applicability domain*.

🧠 **Outliers: handle with care!**
- First check if they're data errors.
- Don't automatically remove them — they might be real but rare cases.
- Some models (trees, SVM) are naturally robust to outliers.
- For sensitive models, use the **spatial sign** transformation.

There are several predictive models that are resistant to outliers, e.g.
- Tree based classification models: create splits of the training set.
- Support Vector Machines (SVM) for classification: disregard a portion of the training set that may be far away from the decision boundary.

If a model is considered to be sensitive to outliers, one data transformation that can minimize the problem is the *spatial sign*. Mathematically, each sample is divided by its squared norm: $$x_{ij}^* = {x_{ij} \over \sqrt{\sum_{j=1}^p x_{ij}^2}}.$$ Since the denominator is intended to measure the squared distance to the center of the predictor's distribution, it is **important** to center and scale the predictor data prior to using this transformation. Note that, unlike centering and scaling, this manipulation of the predictors transform them as a group. Removing predictor variables after applying the spatial sign transformation may be problematic.

🧠 **Spatial sign intuition:** Each data point is projected onto the surface of a unit sphere. Points far from the center get pulled in, reducing outlier influence. Think of it as "normalizing by distance from origin."

🧠 **IMPORTANT:** Always center and scale BEFORE spatial sign, otherwise variables with larger scales dominate.

In [ ]:
# 🧠 TIP: This is a toy example to demonstrate spatial sign.
#   We generate fake data: 1000 points with a linear relationship
#   plus 8 outliers far from the main cluster.
#
#   np.random.normal(mean, std, n) generates random normal values.
#   np.random.uniform(min, max, n) generates random uniform values.
#
# TODO: run this cell as-is to understand the data generation






In [ ]:
# 🧠 TIP: Now we apply spatial sign to bring outliers toward the main cluster.
#
#   Steps:
#   1. Concatenate true data + outliers
#   2. Scale both features (standardize)
#   3. Compute distance from origin for each point
#   4. Divide each coordinate by its distance
#
#   sklearn.preprocessing.scale() centers and scales.
#   np.concatenate() joins arrays.
#
#   NOTE: The original code uses zip() which creates a list of tuples.
#   In Python 3, zip() returns an iterator, so wrap it in list().
#   Or use np.column_stack() instead.
#
# TODO: scale the data, compute spatial sign, plot results





The *spatial sign* transformation brings the outliers towards the majority of the data.

### Data Reduction and Feature Extraction

These methods reduce the data by generating a smaller set of predictors that seek to capture a majority of the information in the original variables. For most data reduction techniques, the new predictors are functions of the original predictors; therefore, all the original predictors are still needed to create the surrogate variables. This class of methods is often called *signal extraction* or *feature extraction* techniques.

🧠 **Dimensionality reduction:** When you have 100+ predictors, many may be redundant or noisy. PCA creates new features (principal components) that capture the most important patterns.
  - Unsupervised: doesn't use the target variable
  - First PC captures the most variance
  - Each subsequent PC captures remaining variance, orthogonal to all previous

Principal component analysis (PCA) seeks to find linear combinations of the predictors, known as principal components (PCs), which capture the most possible variance. The first PC is defined as the linear combination of the predictors that captures the most variability of all possible linear combinations. Then, subsequent PCs are derived such that these linear combinations capture the most remaining variability while also being uncorrelated with all previous PCs. Mathematically,
$$\text{PC}_j = (a_{j1} \times \text{Predictor 1}) + \cdots + (a_{jP} \times \text{Predictor P}).$$
P is the number of predictors. The coefficients $a_{j1}, \cdots, a_{jP}$ are called component weights and help us understand which predictors are most important to each PC.

Let us look at an example from the previous dataset.

In [ ]:
# 🧠 TIP: Select a small subset of predictors for a simple PCA demo.
#   cell_train[['Class', 'FiberWidthCh1', 'EntropyIntenCh1']]
#   We keep 'Class' for coloring the plots.
#
# TODO: select columns 'Class', 'FiberWidthCh1', 'EntropyIntenCh1'
#       from cell_train into a new variable cell_train_subset



In [ ]:
# 🧠 TIP: Scatter plot of the two features, colored by class (PS vs WS).
#   alpha=0.4 makes points semi-transparent (see overlap).
#   s=26 controls marker size.
#
# TODO: create a scatter plot of FiberWidthCh1 vs EntropyIntenCh1
#       color PS blue (squares), WS red (circles)




Calculate PCs

In [ ]:
# 🧠 TIP: sklearn.decomposition.PCA computes principal components.
#   pca.fit() finds the components that maximize variance.
#   pca.explained_variance_ratio_ tells us how much variance each PC captures.
#
#   Here we use only the 2 numeric features (dropping 'Class').
#   With 2 features, we get 2 PCs.
#   The first PC captures ~97% of the variance — almost all the info in 1 dimension!
#
# TODO: import PCA from sklearn.decomposition
# TODO: fit PCA on the 2 feature columns
# TODO: print the explained variance ratios




The first PC summarizes 97% of the original variability, while the second summarizes 3%. Hence, it is reasonable to use only the first PC for modeling since it accounts for the majority of the information in the data.

🧠 **Key insight:** With 2 original features, we can capture 97% of the variance with just 1 PC. This is dimensionality reduction at work!

In [ ]:
# 🧠 TIP: pca.transform() converts original data into PC space.
#   We then plot PC1 vs PC2, colored by class.
#   This is the same data as before, but rotated into PC coordinates.
#
# TODO: use pca.transform() on the feature data
# TODO: scatter plot PC1 vs PC2 colored by class





The primary advantage of PCA is that it creates components that are uncorrelated. PCA preprocessing creates new predictors with desirable characteristics for models that prefer predictors to be uncorrelated.

🧠 **Why uncorrelated matters:** Models like linear regression struggle with multicollinearity (correlated predictors). PCA fixes this by creating orthogonal components.

While PCA delivers new predictors with desirable characteristics, it must be used with understanding and care. PCA seeks predictor-set variation without regard to any further understanding of the predictors (i.e. measurement scales or distributions) or to knowledge of the modeling objectives (i.e. response variable). Hence, without proper guidance, PCA can generate components that summarize characteristics of the data that are irrelevant to the underlying structure of the data and also to the ultimate modeling objectives.

🧠 **PCA caveats:**
- PCA is unsupervised — it doesn't know about your target variable!
- It focuses on variance, not predictive power.
- Always transform skewed predictors and scale BEFORE PCA.
- PLS (partial least squares) is the supervised alternative.

PCA was applied to the entire set of segmentation data predictors.

In [ ]:
# 🧠 TIP: We'll extract all numeric features (excluding metadata columns).
#   cell_train.iloc[:, 4:] selects all columns from index 4 onward.
#   This drops: Unnamed:0, Cell, Case, Class (first 4 columns).
#
# TODO: select columns 4 onwards from cell_train into cell_train_feature
# TODO: display the first 5 rows



Because PCA seeks linear combinations of predictors that maximize variability, it will naturally first be drawn to summarizing predictors that have more variation. If the original predictors are on measurement scales that differ in orders of magnitude or have skewed distributions, PCA will be focusing its efforts on identifying the data structure based on measurement scales and distributional difference rather than based on the important relationships within the data for the current problem. Hence, it is best to first transform skewed predictors and then center and scale the predictors prior to performing PCA.

🧠 **Proper PCA workflow:**
  1. Identify positive predictors → apply Box-Cox to fix skewness
  2. Separate non-positive predictors (can't Box-Cox these)
  3. Concatenate transformed + non-positive
  4. Scale everything (center + unit variance)
  5. Now PCA is meaningful!

In [ ]:
# 🧠 TIP: We apply Box-Cox to ALL positive predictors.
#   cell_train_feature.apply(lambda x: np.all(x > 0)) checks if ALL values > 0.
#   np.where() gives the indices where this is True.
#
#   For positive features: apply boxcox to each column.
#   For non-positive features: keep as-is.
#   Then concatenate with np.c_[].
#
#   NOTE: original code has a typo 'npn-positive' — it means 'non-positive'
#
# TODO: find positive columns, transform them with Box-Cox
# TODO: concatenate with non-positive columns
# TODO: print counts of positive and non-positive features





In [ ]:
# 🧠 TIP: sklearn.preprocessing.scale() centers (subtract mean) and scales (divide by std).
#   with_mean=True, with_std=True means both centering and scaling.
#   After this, all features have mean=0 and variance=1.
#
# TODO: scale the transformed features




The second caveat of PCA is that it does not consider the modeling objective or response variable when summarizing variability -- it is an *unsupervised technique*. If the predictive relationship between the predictors and response is not connected to the predictors' variability, then the derived PCs will not provide a suitable relationship with the response. In this case, a *supervised technique*, like PLS will derive components while simultaneously considering the corresponding response.

🧠 **PCA vs PLS:** PCA finds components that explain predictor variance. PLS finds components that explain predictor variance AND predict the response. If you care about prediction, PLS is often better.

To decide how many components to retain after PCA, a heuristic approach is to create a scree plot, which contains the ordered component number (x-axis) and the amount of summarized variability (y-axis). Generally, the component number prior to the tapering off of variation is the maximal component that is retained. In an automated model building process, the optimal number of components can be determined by cross-validation.

🧠 **Scree plot:** Plot variance explained vs component number. Look for the "elbow" where the curve flattens — that's usually the right number of components to keep.

In [ ]:
# 🧠 TIP: Fit PCA on the full (transformed + scaled) training data.
#   Then plot the explained variance ratio for all components.
#   Look at the y-axis — each bar shows how much variance that PC captures.
#
# TODO: fit PCA on cell_train_feature_tr
# TODO: plot the explained variance ratio




In [ ]:
# 🧠 TIP: Look at the first 4 components specifically.
#   With ~116 features, the first 4 capture ~42% of total variance.
#   This is common for high-dimensional data — variance is spread thin.
#
# TODO: print the explained variance ratio for first 4 components
# TODO: print the cumulative sum



Visually examining the principal components is a critical step for assessing data quality and gaining intuition for the problem. To do this, the first few PCs can be plotted against each other and the plot symbols can be colored by the relevant characteristics, such as the class labels. If PCA has captured a sufficient amount of the information in the data, this type of plot can demonstrate clusters of samples or outliers that may prompt a closer examination of the individual data points. Note that the scale of the components tend to become smaller as they account for less and less variation in the data. If axes are displayed on separate scales, there is the potential to over-interpret any patterns that might be seen for components that account for small amounts of variation.

In [ ]:
# 🧠 TIP: Fit PCA with 3 components and transform the data.
#   Then create a scatter plot matrix (3 rows, 3 cols) showing PCs vs each other.
#   Color by class to see if cells separate by segmentation quality.
#
# TODO: create a 3-component PCA, fit and transform the training data
# TODO: create a 3x3 grid of scatter plots (PC1/2/3 vs each other)
#       colored by PS (blue) vs WS (red)




# PC1 vs PC3

# PC2 vs PC3

# PC1 vs PC2



Since the percentages of variation explained are not large for the first three components, it is important not to over-interpret the resulting image. From this plot, there appears to be some separation between the classes when plotting the first and second components. However, the distribution of the well-segmented cells is roughly contained within the distribution of the poorly identified cells. One conclusion is that the cell types are not easily separated.

🧠 **What this tells us:** The classes (PS vs WS) overlap a lot in PC space, meaning they're not easily separable. This is a hard classification problem!

Another exploratory use of PCA is characterizing which predictors are associated with each component. Recall that each component is a linear combination of the predictors and the coefficient for each predictor is called the loading. Loadings close to zero indicate that the predictor variable did not contribute much to that component.

🧠 **Loadings = feature importance for each PC.** A large positive or negative loading means that predictor strongly influences that component.

In [ ]:
# 🧠 TIP: pca.components_ has shape (n_components, n_features).
#   Each row gives the loadings for one PC.
#   Here it's (3, 116) — 3 PCs, each with 116 loadings.
#
# TODO: check the shape of pca.components_



## 3.4 Dealing with Missing Values

In many cases, some predictors have no values for a given sample. It is important to understand *why* the values are missing. First and foremost, it is important to know if the pattern of missing data is related to the outcome. This is called *informative missingness* since the missing data pattern is instructional on its own. Informative missingness can induce significant bias in the model.

🧠 **Missing data types:**
- **MCAR** (Missing Completely at Random): no pattern, just random gaps.
- **MAR** (Missing at Random): depends on observed data.
- **MNAR** (Missing Not at Random): depends on unobserved data (hardest to handle).

If the *fact* that a value is missing tells you something about the outcome, that's **informative missingness** — and it can bias your model.

Missing data should not be confused with *censored* data where the exact value is missing but something is known about its value. When building traditional statistical models focused on interpretation or inference, the censoring is usually taken into account in a formal manner by making assumptions about the censoring mechanism. For predictive models, it is more common to treat these data as simple missing data or use the censored value as the observed value.

Missing values are more often related to predictive variables than the sample. Because of this, amount of missing data may be concentrated in a subset of predictors rather than occurring randomly across all the predictors. In some cases, the percentage of missing data is substantial enough to remove this predictor from subsequent modeling activities.

🧠 **First question to ask:** Is the missingness concentrated in specific columns (predictors) or specific rows (samples)?
- If a column is mostly missing → remove it.
- If a row is mostly missing → remove it (if you have enough data).

There are cases where the missing values might be concentrated in specific samples. For large datasets, removal of samples based on missing values is not a problem, assuming that the missingness is not informative. In smaller datasets, there is a steep price in removing samples; some of alternative approaches described below may be more appropriate.

If we do not remove the missing data, there are two general approaches. First, a few predictive models, especially tree-based techniques, can specifically account for missing data. Alternatively, missing data can be imputed. In this case, we can use information in the training set predictors to, in essence, estimate the values of other predictors.

Imputation is just another layer of modeling where we try to estimate values of the predictor variables based on other predictor variables. The most relevant scheme for accomplishing this is to use the training set to build an imputation model for each predictor in the data set. Prior to model training or the prediction of new samples, missing values are filled in using imputation. Note that this extra layer of models adds uncertainty. If we are using resampling to select tuning parameter values or to estimate performance, the imputation should be incorporated within the resampling. This will increase the computational time for building models, but it will also provide honest estimates of model performance.

If the number of predictors affected by missing values is small, an exploratory analysis of the relationships between the predictors is a good idea. For example, visualization or methods like PCA can be used to determine if there are strong relationships between the predictors. If a variable with missing values is highly correlated with another predictor that has few missing values, a focused model can often be effective for imputation.

One popular technique for imputation is a $K$-nearest neighbor model. A new sample is imputed by finding the samples in the training set "closest" to it and averages these nearby points to fill in the value. One advantage of this approach is that the imputed data are confined to be within the range of the training set values. One disadvantage is that the entire training set is required every time a missing value needs to be imputed. Also, the number of neighbors is a tuning parameter, as is the method for determining "closeness" of two points. However, Troyanskaya et al. (2001) found the nearest neighbor approach to be fairly robust to the tuning parameters, as well as the amount of missing data.

🧠 **KNN Imputation:**
- Find the K most similar samples (nearest neighbors).
- Average their values to fill in the missing one.
- Works well when data has local structure.
- Downside: computationally expensive for large datasets.

In [ ]:
# 🧠 TIP: We'll simulate imputation by hiding 'VarIntenCh3' values.
#   We randomly sample 50 test set rows.
#   Then we'll try to predict VarIntenCh3 using other features.
#
#   random.sample(range(n), 50) picks 50 random indices.
#   np.sort() ensures proper order.
#
#   We create:
#   - cell_test_subset_f: all OTHER features (drop VarIntenCh3)
#   - cell_test_subset_v: the actual VarIntenCh3 values (to compare)
#   - cell_train_f: training features (without VarIntenCh3)
#   - cell_train_v: training VarIntenCh3 values
#
#   NOTE: df.drop('col', 1) drops a column. axis=1 means columns.
#   df.drop('col', axis=1) is the modern syntax.
#
# TODO: randomly sample 50 test set rows for imputation demo
# TODO: separate features and target (VarIntenCh3)




In [ ]:
# 🧠 TIP: StandardScaler standardizes features (z-score).
#   IMPORTANT: Fit on TRAINING data, transform BOTH train and test.
#   This prevents data leakage from test set.
#
#   sc_f.fit_transform(train) → learns mean/std and transforms
#   sc_f.transform(test) → uses TRAINING mean/std to transform test
#
# TODO: create StandardScaler for features (sc_f) and target (sc_v)
# TODO: fit_transform on training, transform on test





In [ ]:
# 🧠 TIP: NearestNeighbors finds the K closest points.
#   n_neighbors=5 means we look at 5 nearest training samples.
#   Then we average their VarIntenCh3 values as our imputed value.
#
#   NOTE: We skip the first neighbor (index[0]) because that's the
#   point itself (if it happened to be in the training set).
#   We use indices[1:] instead.
#
# TODO: fit NearestNeighbors on training features
# TODO: find neighbors for each test point
# TODO: impute by averaging neighbors' VarIntenCh3 values





Find the predictor with highest correlation.

In [ ]:
# 🧠 TIP: Pearson correlation measures linear relationship.
#   scipy.stats.pearsonr(x, y) returns (correlation, p-value).
#   A correlation of 0.89 is very strong!
#
#   'DiffIntenDensityCh3' is highly correlated with 'VarIntenCh3'.
#   This makes it useful for linear model imputation.
#
# TODO: compute Pearson correlation between VarIntenCh3 and DiffIntenDensityCh3




In [ ]:
# 🧠 TIP: Linear regression imputation uses a correlated predictor.
#   Train: VarIntenCh3 ~ DiffIntenDensityCh3
#   Predict: use the trained model on test data.
#
#   NOTE: The original code accesses column index by name.
#   cell_train_f.columns.get_loc('col_name') returns the integer index.
#
#   We reshape to (n, 1) because sklearn expects 2D input for a single feature.
#
# TODO: fit a linear regression using DiffIntenDensityCh3 to predict VarIntenCh3
# TODO: predict on test set




Correlation between the real and imputed values

In [ ]:
# 🧠 TIP: Compare imputation quality using Pearson correlation.
#   Higher correlation = better imputation.
#   Both KNN (~0.87) and Linear Model (~0.86) perform well here.
#   Linear model is simpler and slightly better for THIS case,
#   because there's one very highly correlated predictor.
#   KNN is generally more robust across many scenarios.
#
# TODO: compute and print correlations for both imputation methods



Note that the better performance of linear model is because of the high correlation (0.895) between these two predictors. kNN is generally more robust since it takes all predictors into consideration.

🧠 **Imputation summary:**
- KNN: Uses all features, more robust generally.
- Linear model: Simple, effective when a single correlated predictor exists.
- Both require scaling first!
- Both should be cross-validated within the modeling pipeline.

In [ ]:
# 🧠 TIP: Scatter plots of original vs imputed values.
#   Points should fall near the dashed diagonal line if imputation is good.
#
# TODO: create side-by-side scatter plots for KNN and LM imputation






## 3.5 Removing Predictors

There are potential advantages to removing predictors prior to modeling. First, fewer predictors means decreased computational time and complexity. Second, if two predictors are highly correlated, this implies that they are measuring the same underlying information. Removing one should not compromise the performance of the model and might lead to a more parsimonious and interpretable model. Third, some models can be crippled by predictors with degenerate distributions, e.g. near-zero variance predictors. In these cases, there can be a significant improvement in model performance and/or stability without the problematic variables.

🧠 **Why remove predictors?**
1. Speed: fewer features = faster training
2. Redundancy: correlated features add noise, not signal
3. Degeneracy: near-zero variance features are useless
4. Interpretability: simpler models are easier to understand

A rule of thumb for detecting near-zero variance predictors:
- The fraction of unique values over the sample size is low (say 10%)
- The ratio of the frequency of the most prevalent value to the frequency of the second most prevalent value is large (say around 20)

If both of these criteria are true and the model in question is susceptible to this type of predictor, it may be advantageous to remove the variable from the model.

🧠 **Near-zero variance:** If a predictor has almost the same value for every sample, it carries almost no information. Example: a column with 99% zeros and 1% ones.

### Between-Predictor Correlations

*Collinearity* is the technical term for the situation where a pair of predictor variables have a substantial correlation with each other. It is also possible to have relationships between multiple predictors at once (called *multicollinearity*).

🧠 **Multicollinearity problem:** When predictors are highly correlated, it becomes hard to tell which one is actually driving the prediction. Coefficients become unstable and hard to interpret.

A direct visualization of the correlation matrix from the training set.

In [ ]:
# 🧠 TIP: The original notebook likely creates a correlation matrix heatmap here.
#   .corr() computes pairwise correlations between ALL numeric columns.
#   A heatmap visualizes these correlations as colors.
#
#   Dark red = strong positive correlation (they move together).
#   Dark blue = strong negative correlation (one goes up, other down).
#   White/light = no correlation.
#
#   This helps identify redundant predictors to remove.
#
#   If this code isn't in your version, here's how you'd do it:
#
#   import seaborn as sns
#   corr_matrix = cell_train_feature.corr()
#   sns.heatmap(corr_matrix)
#
# TODO (optional): create a heatmap of the correlation matrix

# Your code here (or skip if not in original notebook)

---
## 🧠 Chapter 3 Summary: The Pre-processing Pipeline

**The full workflow for preparing predictors:**

1. **Load & Explore** — shape, head, group counts
2. **Handle Missing Values** — remove or impute (KNN, linear model)
3. **Fix Skewness** — Box-Cox transformation for positive predictors
4. **Center & Scale** — StandardScaler (fit on train, transform both)
5. **Handle Outliers** — spatial sign transformation if needed
6. **Remove Problematic Predictors** — near-zero variance, high correlation
7. **Reduce Dimensionality** — PCA (after scaling!) or PLS
8. **Verify** — check scree plot, loadings, PC scatter plots

🧠 **Golden rules:**
- Fit transformations ONLY on training data, apply to all data
- Always scale before PCA
- Always fix skewness before scaling
- Handle missing data BEFORE training
- Cross-validate imputation within the modeling pipeline